<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

# Function Calling — Phase 2
## Tool Definition & Schema Design

**هدف فاز:** طراحی قراردادهایی که به LLM معرفی می‌کنیم تا ابزارها را درست تشخیص دهد و آرگومان‌های مناسب تولید کند.

در این فاز Name، Description، JSON Schema، انواع داده، پارامترهای اجباری و اختیاری، محدودیت‌ها، Strict Mode و اعتبارسنجی با Pydantic را یاد می‌گیریم.

> مثال‌های اجرایی با Python و Ollama هستند. قابلیت‌هایی مانند Strict Mode وابسته به Provider هستند و نباید آن‌ها را به همه مدل‌ها تعمیم داد.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## فهرست مطالب

1. Defining Functions as Tools
2. Function Names and Descriptions
3. JSON Schema Fundamentals
4. Parameters and Data Types
5. Required vs Optional Parameters
6. Enum and Constraints
7. Strict Mode and Schema Validation
8. Designing Good Tool Schemas
9. Tool Descriptions and Prompt Engineering
10. Practical HR Tool Design

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 1. Defining Functions as Tools

وجود یک تابع Python به‌تنهایی به این معنا نیست که LLM می‌تواند آن را فراخوانی کند. باید قرارداد ابزار را به مدل معرفی کنیم.

Tool Definition معمولاً سه بخش اصلی دارد:

- **Name:** شناسه ابزار
- **Description:** شرح مسئولیت ابزار و زمان مناسب استفاده
- **Parameters:** ورودی‌ها، نوع داده‌ها و محدودیت‌ها

تابع واقعی:

```python
def get_leave_balance(employee_id: str) -> dict:
    ...
```

تعریف ابزار:

```json
{
  "type": "function",
  "function": {
    "name": "get_leave_balance",
    "description": "Get the remaining annual leave balance for an employee.",
    "parameters": {
      "type": "object",
      "properties": {
        "employee_id": {"type": "string"}
      },
      "required": ["employee_id"]
    }
  }
}
```

این JSON تابع را اجرا نمی‌کند؛ فقط ابزار و قرارداد ورودی آن را به مدل معرفی می‌کند.

| بخش | مسئولیت |
|---|---|
| Tool Definition | معرفی ابزار به مدل |
| Python Function | اجرای منطق واقعی |
| Application | اتصال Tool Call به تابع و کنترل اجرا |

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 2. Function Names and Descriptions

نام ابزار باید دقیق، توصیفی و سازگار با نام‌گذاری بقیه ابزارها باشد.

| نام ضعیف | نام بهتر |
|---|---|
| `leave` | `get_leave_balance` |
| `employee` | `get_employee_profile` |
| `request` | `create_leave_request` |
| `data` | `search_company_policies` |

Description باید روشن کند ابزار چه کاری انجام می‌دهد، چه زمانی باید استفاده شود و چه کاری انجام نمی‌دهد.

توضیح ضعیف:

```text
Gets leave.
```

توضیح بهتر:

```text
Retrieve the remaining annual leave balance for a specific employee.
Use this tool when the user asks how many annual leave days an employee
has available. This tool does not create or modify leave requests.
```

ابزارها را تک‌مسئولیتی طراحی کن. برای مثال `get_leave_balance` و `create_leave_request` دو قابلیت مستقل‌اند و بهتر است ابزارهای جداگانه باشند.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 3. JSON Schema Fundamentals

JSON Schema روشی استاندارد برای توصیف ساختار و محدودیت‌های داده JSON است. در Function Calling از آن برای تعریف پارامترهای ابزار استفاده می‌شود.

```json
{
  "type": "object",
  "properties": {
    "employee_id": {
      "type": "string",
      "description": "The unique employee identifier."
    }
  },
  "required": ["employee_id"]
}
```

- `type`: نوع داده
- `properties`: فیلدهای Object
- `required`: فیلدهای اجباری
- `description`: توضیح معنایی فیلد

برای ساختار تو در تو از Object و برای فهرست مقادیر از Array استفاده می‌کنیم.

```json
{
  "type": "object",
  "properties": {
    "period": {
      "type": "object",
      "properties": {
        "start_date": {"type": "string"},
        "end_date": {"type": "string"}
      },
      "required": ["start_date", "end_date"]
    },
    "dates": {
      "type": "array",
      "items": {"type": "string"}
    }
  },
  "required": ["period"]
}
```

همه Providerها لزوماً تمام قابلیت‌های JSON Schema را در Tool Calling پشتیبانی نمی‌کنند؛ Schema قابل استفاده را با مستندات Provider تطبیق بده.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 4. Parameters and Data Types

نوع داده باید با معنای واقعی پارامتر هماهنگ باشد.

| نوع | کاربرد | نمونه |
|---|---|---|
| `string` | متن و شناسه | شناسه کارمند |
| `integer` | عدد صحیح | تعداد روز |
| `number` | عدد صحیح یا اعشاری | مبلغ |
| `boolean` | درست/نادرست | فعال بودن |
| `object` | ساختار چندفیلدی | بازه تاریخ |
| `array` | فهرست | شناسه‌ها |

```json
{
  "type": "object",
  "properties": {
    "employee_id": {"type": "string"},
    "days": {"type": "integer"},
    "is_half_day": {"type": "boolean"},
    "estimated_cost": {"type": "number"},
    "dates": {
      "type": "array",
      "items": {"type": "string"}
    }
  },
  "required": ["employee_id", "days"]
}
```

شناسه `00124` را بهتر است `string` نگه داریم، چون شناسه برای محاسبه عددی نیست و تبدیل به عدد ممکن است صفر ابتدایی را حذف کند.

برای تاریخ می‌توان از رشته با قالب مشخص استفاده کرد. `format: date` در JSON Schema تعریف شده، اما enforce شدن آن به Validator و Provider وابسته است.

```json
{"type": "string", "format": "date"}
```

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 5. Required vs Optional Parameters

آرایه `required` مشخص می‌کند کدام Propertyها باید در Object وجود داشته باشند.

در مثال زیر `employee_id` اجباری و `status` اختیاری است:

```json
{
  "type": "object",
  "properties": {
    "employee_id": {"type": "string"},
    "status": {"type": "string"}
  },
  "required": ["employee_id"]
}
```

**Optional با Nullable متفاوت است:**

- Optional: فیلد می‌تواند اصلاً وجود نداشته باشد.
- Nullable: فیلد وجود دارد، اما مقدارش می‌تواند `null` باشد.

نمونه Nullable در JSON Schema:

```json
{
  "type": "object",
  "properties": {
    "status": {"type": ["string", "null"]}
  },
  "required": ["status"]
}
```

در این نمونه فیلد باید وجود داشته باشد، اما مقدار آن رشته یا null است. رفتار مورد انتظار را در Backend نیز صریح تعریف کن.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 6. Enum and Constraints

برای محدودکردن مقادیر می‌توان از Constraintهای JSON Schema استفاده کرد.

**Enum:**

```json
{"type": "string", "enum": ["pending", "approved", "rejected"]}
```

**محدوده عددی:**

```json
{"type": "integer", "minimum": 1, "maximum": 30}
```

**طول رشته:**

```json
{"type": "string", "minLength": 3, "maxLength": 20}
```

**الگوی رشته با Regex:**

```json
{"type": "string", "pattern": "^E-[0-9]{3}$"}
```

این قواعد Schema برای راهنمایی و محدودسازی ورودی مفیدند، اما مرز امنیتی محسوب نمی‌شوند. Application باید ورودی را مستقل اعتبارسنجی کند.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 7. Strict Mode and Schema Validation

Strict Mode در برخی APIها امکان محدودکردن آرگومان‌های تولیدشده به Schema تعریف‌شده را فراهم می‌کند.

**Provider-specific:** در OpenAI Function Calling پارامتر `strict` وجود دارد و قواعد Schema خاص خود را دارد. این قابلیت را نباید بدون پشتیبانی مستندات به Ollama یا همه Providerها تعمیم داد.

حتی در حالت Strict نیز Backend باید نام ابزار، ورودی‌ها، مجوزها و قواعد کسب‌وکار را بررسی کند.

### اعتبارسنجی با Pydantic

Pydantic کتابخانه‌ای برای تعریف مدل داده و اعتبارسنجی آن در Python است.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## نصب Pydantic

```python
%pip install -U pydantic
```

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## نمونه اعتبارسنجی با Pydantic

```python
from pydantic import BaseModel, Field, ValidationError

class LeaveBalanceInput(BaseModel):
    employee_id: str = Field(
        min_length=5,
        max_length=20,
        description="The unique employee identifier."
    )

valid_payload = {"employee_id": "E-104"}
invalid_payload = {"employee_id": ""}

validated = LeaveBalanceInput.model_validate(valid_payload)
print(validated.model_dump())

try:
    LeaveBalanceInput.model_validate(invalid_payload)
except ValidationError as exc:
    print(exc)
```

Pydantic می‌تواند شکل و محدودیت داده را بررسی کند؛ اما وجود کارمند در دیتابیس و مجوز مشاهده اطلاعات باید جداگانه در Backend بررسی شود.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 8. Designing Good Tool Schemas

یک Schema خوب باید هم برای مدل قابل فهم باشد و هم برای Application قابل اعتبارسنجی.

**۱. یک ابزار، یک مسئولیت روشن**

```python
get_leave_balance(...)
create_leave_request(...)
cancel_leave_request(...)
get_leave_request_status(...)
```

**۲. از نام پارامترهای مبهم پرهیز کن.** به‌جای `data` یا `value` از نام‌هایی مانند `employee_id`، `start_date` و `end_date` استفاده کن.

**۳. ورودی غیرضروری نخواه.** برای عملیات حساس، شناسه کاربر احراز هویت‌شده را بهتر است Backend از Context امن درخواست استخراج کند، نه اینکه به مدل اعتماد کند.

**۴. خروجی ابزار را نیز قراردادی طراحی کن.** نمونه موفق:

```json
{
  "success": true,
  "employee_id": "E-104",
  "remaining_days": 12
}
```

نمونه خطا:

```json
{
  "success": false,
  "error_code": "EMPLOYEE_NOT_FOUND",
  "message": "Employee was not found."
}
```

این‌ها قرارداد خروجی Application هستند و با Schema ورودی Tool تفاوت دارند.

**۵. Schema را بیش از حد پیچیده نکن.** فقط پارامترهای لازم را تعریف کن.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 9. Tool Descriptions and Prompt Engineering

Schema ساختار داده را مشخص می‌کند؛ Description به مدل کمک می‌کند معنای ابزار و زمان استفاده از آن را بفهمد.

```text
get_employee_profile:
Retrieve basic profile information for an employee.
Use when the user asks about an employee's profile or department.

get_leave_balance:
Retrieve the remaining annual leave balance for an employee.
Use when the user asks about available annual leave days.
```

Description ابزار با System Prompt یکی نیست:

- Tool Description: توضیح یک ابزار مشخص.
- System Prompt: دستورالعمل کلی رفتار مدل.

هر دو می‌توانند رفتار مدل را هدایت کنند، اما جایگزین یکدیگر نیستند.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## 10. Practical Example — طراحی ابزارهای HR

در این مثال سه ابزار طراحی می‌کنیم:

1. `get_leave_balance`
2. `create_leave_request`
3. `search_leave_requests`

این بخش روی طراحی قرارداد و اعتبارسنجی ورودی تمرکز دارد؛ اجرای کامل Tool Calling و Tool Loop در فازهای بعدی توسعه داده می‌شود.

</div>

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

### 10.1 تعریف Tool Schemas

```python
hr_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_leave_balance",
            "description": (
                "Retrieve the remaining annual leave balance for an employee. "
                "Use when the user asks about available annual leave days. "
                "This tool only reads data."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {
                        "type": "string",
                        "description": "The unique employee identifier."
                    }
                },
                "required": ["employee_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_leave_request",
            "description": (
                "Create an annual leave request. Use only when the user "
                "explicitly asks to submit a leave request."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"},
                    "start_date": {
                        "type": "string",
                        "description": "Date in YYYY-MM-DD format."
                    },
                    "end_date": {
                        "type": "string",
                        "description": "Date in YYYY-MM-DD format."
                    },
                    "reason": {"type": "string"}
                },
                "required": ["employee_id", "start_date", "end_date"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_leave_requests",
            "description": "Search leave requests for a specific employee.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"},
                    "status": {
                        "type": "string",
                        "enum": ["pending", "approved", "rejected"]
                    }
                },
                "required": ["employee_id"]
            }
        }
    }
]

print(f"Defined {len(hr_tools)} tools.")
```

</div>

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

### 10.2 بررسی Tool Definition

```python
import json

print(json.dumps(hr_tools, indent=2, ensure_ascii=False))
```

</div>

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

### 10.3 مدل‌های ورودی با Pydantic

```python
from datetime import date
from typing import Literal

from pydantic import BaseModel, Field, model_validator


class GetLeaveBalanceInput(BaseModel):
    employee_id: str = Field(min_length=5, max_length=20)


class CreateLeaveRequestInput(BaseModel):
    employee_id: str = Field(min_length=5, max_length=20)
    start_date: date
    end_date: date
    reason: str | None = Field(default=None, max_length=500)

    @model_validator(mode="after")
    def validate_date_range(self):
        if self.end_date < self.start_date:
            raise ValueError("end_date must not be earlier than start_date")
        return self


class SearchLeaveRequestsInput(BaseModel):
    employee_id: str = Field(min_length=5, max_length=20)
    status: Literal["pending", "approved", "rejected"] | None = None
```

</div>

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

### 10.4 اعتبارسنجی ورودی‌های معتبر

```python
valid_balance_input = {"employee_id": "E-104"}

valid_leave_request = {
    "employee_id": "E-104",
    "start_date": "2026-11-20",
    "end_date": "2026-11-22",
    "reason": "Personal trip"
}

valid_search = {
    "employee_id": "E-104",
    "status": "pending"
}

balance = GetLeaveBalanceInput.model_validate(valid_balance_input)
leave_request = CreateLeaveRequestInput.model_validate(valid_leave_request)
search = SearchLeaveRequestsInput.model_validate(valid_search)

print(balance.model_dump())
print(leave_request.model_dump(mode="json"))
print(search.model_dump())
```

</div>

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

### 10.5 بررسی ورودی نامعتبر

```python
invalid_examples = [
    (
        CreateLeaveRequestInput,
        {
            "employee_id": "E-104",
            "start_date": "2026-11-25",
            "end_date": "2026-11-20"
        }
    ),
    (
        SearchLeaveRequestsInput,
        {
            "employee_id": "E-104",
            "status": "waiting"
        }
    )
]

for model, payload in invalid_examples:
    try:
        model.model_validate(payload)
    except Exception as exc:
        print(f"{model.__name__}:")
        print(exc)
        print("-" * 50)
```

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

### 10.6 تولید JSON Schema از Pydantic

```python
print(
    json.dumps(
        CreateLeaveRequestInput.model_json_schema(),
        indent=2,
        ensure_ascii=False
    )
)
```

Pydantic می‌تواند JSON Schema تولید کند. با این حال، Schema تولیدشده را باید با زیرمجموعه قابل پشتیبانی Provider تطبیق داد؛ سازگاری خودکار با همه APIها تضمین‌شده نیست.

در این مثال Tool Schema و مدل Pydantic جدا تعریف شده‌اند تا نقش هر دو مشخص شود. در پروژه واقعی، با تست یا منبع مشترک از اختلاف تدریجی این قراردادها جلوگیری کن.

</div>

<div class="markdown-content" dir="rtl" style="direction: rtl; text-align: right;">

<style>
.markdown-content { direction: rtl; text-align: right; }
.markdown-content pre, .markdown-content pre code { direction: ltr !important; text-align: left !important; unicode-bidi: isolate; }
.markdown-content code { direction: ltr; unicode-bidi: isolate; }
.markdown-content table { direction: rtl; }
</style>

## جمع‌بندی فاز دوم

- Tool Definition قرارداد معرفی ابزار به LLM است.
- Name باید هدف ابزار را روشن کند.
- Description باید کاربرد و محدوده ابزار را توضیح دهد.
- JSON Schema ساختار و محدودیت پارامترها را توصیف می‌کند.
- `required` فیلدهای اجباری را مشخص می‌کند.
- Optional بودن با Nullable بودن تفاوت دارد.
- `enum`، `minimum`، `maximum`، `minLength`، `maxLength` و `pattern` برای محدودسازی ورودی‌اند.
- Strict Mode وابسته به Provider است.
- اعتبارسنجی Backend همچنان ضروری است.
- ابزارها را با مسئولیت روشن و پارامترهای حداقلی طراحی کن.

## منابع رسمی

- [Ollama — Tool Calling](https://docs.ollama.com/capabilities/tool-calling)
- [Ollama Python Library](https://github.com/ollama/ollama-python)
- [OpenAI — Function Calling](https://developers.openai.com/api/docs/guides/function-calling)
- [OpenAI — Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [JSON Schema — Getting Started](https://json-schema.org/learn/getting-started-step-by-step)
- [Pydantic — JSON Schema](https://docs.pydantic.dev/latest/concepts/json_schema/)
- [Pydantic — Models](https://docs.pydantic.dev/latest/concepts/models/)

</div>

In [ ]:
%pip install -U ollama pydantic

In [2]:
from ollama import chat

MODEL_NAME = "qwen3:4b"

response = chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)

print(response.message.content)

Hello!


In [3]:
from datetime import date
from typing import Literal
from pydantic import BaseModel, Field, model_validator

class GetLeaveBalanceInput(BaseModel):
    employee_id: str = Field(min_length=5, max_length=20)

class CreateLeaveRequestInput(BaseModel):
    employee_id: str = Field(min_length=5, max_length=20)
    start_date: date
    end_date: date
    reason: str | None = Field(default=None, max_length=500)

    @model_validator(mode="after")
    def validate_date_range(self):
        if self.end_date < self.start_date:
            raise ValueError("end_date must not be earlier than start_date")
        return self

class SearchLeaveRequestsInput(BaseModel):
    employee_id: str = Field(min_length=5, max_length=20)
    status: Literal["pending", "approved", "rejected"] | None = None